# Week 3 SQL Assignment

## Data Setup

### Objective

The objective of this notebook is to import the Superstore dataset and create the required database tables for further SQL analysis.

## Step 1 : Import Required Libraries

In this step, the required Python libraries are imported to perform data loading and database operations.

These libraries will be used throughout the notebook for creating and managing the SQLite database.

- pandas is used to read and manage the dataset.
- sqlite3 is used to create and work with the SQLite database.

In [1]:
# Import pandas library for reading and manipulating the dataset
import pandas as pd

#Import sqlite3 library for database operations
import sqlite3

In [2]:
# Close any previous database connection

try:
    conn.close()
except:
    pass

## Step 2 : Load Dataset

Now we will load the Sample Superstore Dataset into a pandas DataFrame so that it can be used for SQL analysis.

In [3]:
# Read the dataset using latin1 encoding
df = pd.read_csv("../data/Sample_Superstore.csv", encoding="latin1")

## Step 3 : Display First Five Records.

The head() function is used to verify that the dataset has been loaded successfully.

In [4]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


### Observation

The dataset has been loaded successfully into a pandas Dataframe.

The first five rows provide an overview of customer, order and product information in the dataset.

This confirms that the dataset is database creation and further SQL analysis.

## Step 4 : Display all column names.

This helps in understanding the dataset structure before creating tables.

In [5]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')

### Observation

The dataset contains customer, product and order related attributes.
The presence of separate customer, product and order fields makes the dataset suitable for normalization into multiple tables.

These columns will be used to create normalized tables.

## Step 5 : Create SQLite Database Connection 

Creating a database connection allows us to store and query data using SQL commands.

In [6]:
# Create a new database connection

conn = sqlite3.connect("../superstore.db", timeout=30)

cursor = conn.cursor()

## Step 6 : Create Raw Table

The complete dataset will be stored in a table named superstore_raw.

This raw table will act as the source table for creating customers, orders and products tables.

In [7]:
# Remove existing raw table if present

cursor.execute(

'''

DROP TABLE IF EXISTS superstore_raw

'''

)

conn.commit()

In [8]:
df.to_sql(
    "superstore_raw",
    conn,
    if_exists="replace",
    index=False
)

9994

### Observation 

The raw dataset has been successfully stored inside the SQLite database.

This table will be used as the base table for creating normalized tables in the next step.

## Step 7 : Verify the raw table.

In [9]:
pd.read_sql(
    "SELECT * FROM superstore_raw LIMIT 5",
    conn
)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


### Observation

The first five records of the superstore_raw table are displayed successfully.

This verifies that the data has been imported into the SQLite database without any issues.

## Step 8 : Create Customers Table

The raw dataset contains repeated customer information because one customer can place multiple orders.

To avoid storing duplicate customer records, we create a separate customers table.

This process is called normalization, where related information is stored in separate tables to reduce redundancy and improve data organisation.

### Customer Table Structure

The customers table will contain the following attributes:

- Customer ID
- Customer Name
- Segment
- Country
- City
- State
- Postal Code
- Region

In [10]:
# SQL query to create the customers table

create_customers = """

CREATE TABLE IF NOT EXISTS customers (
    
    customer_id TEXT,
    customer_name TEXT,
    segment TEXT,
    country TEXT,
    city TEXT,
    state TEXT,
    postal_code INTEGER,
    region TEXT
    
);

"""

cursor.execute(create_customers)

conn.commit()

print("Customers table created successfully.")


Customers table created successfully.


### EXPLANATION

The CREATE TABLE statement is used to create a new table in the database.

Each column stores a specific type of customer information.

This table will contain only customer-related details.Separating customer information improves database consistency and avoids storing repeated customer details in every order record.

## Step 9 : Insert Data into Customers Table

The customer information is currently available inside the superstore_raw table.

Since the same customer may appear multiple times, we use the DISTINCT keyword to insert only unique customer records into the customers table.

In [11]:
conn.execute("""

INSERT INTO customers

SELECT DISTINCT

    "Customer ID",

    "Customer Name",

    Segment,

    Country,

    City,

    State,

    "Postal Code",

    Region

FROM superstore_raw;

""")

### Why DISTINCT:

DISTINCT removes duplicate rows.

Without DISTINCT, the same customer would be inserted multiple times because a customer can place many orders.

Using DISTINCT ensures that each customer appears only once in the customers table.

In [12]:
pd.read_sql(
   
    """
    
    SELECT *
    
    FROM customers
    
    LIMIT 5;
    
    """,
    conn
)

,customer_id,customer_name,segment,country,city,state,postal_code,region
0,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South
1,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West
2,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South
3,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West
4,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,28027,South


### Observation

The customers table has been created successfully.

Only unique customer records have been inserted using the DISTINCT keyword.

This reduces data redundancy and improves database organisation.

## Step 10 : Create Products Table

The raw dataset also contains product-related information.

Instead of storing product details repeatedly for every order, we create a separate products table.

In [13]:
# SQL query to create products table

create_products = """

CREATE TABLE IF NOT EXISTS products(

    product_id TEXT,

    category TEXT,

    sub_category TEXT,

    product_name TEXT

);

"""

conn.execute(create_products)

conn.commit()

print("Products table created successfully.")

Products table created successfully.


In [14]:
# Insert Data

conn.execute("""

INSERT INTO products

SELECT DISTINCT

    "Product ID",

    Category,

    "Sub-Category",

    "Product Name"

FROM superstore_raw;

""")


In [15]:
# Save the inserted records

conn.commit()

In [16]:
# Verification

pd.read_sql(

"""

SELECT *

FROM products

LIMIT 5;

""",

conn)

,product_id,category,sub_category,product_name
0,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase
1,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,..."
2,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...
3,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table
4,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System


### Observation

The products table stores product-specific information independently.

Using a separate products table minimizes redundancy and improves the overall database structure.

The DISTINCT keyword ensures that duplicate product records are removed.


## Step 11 : Create Orders Table

The orders table contains transaction-related information.

Each row represents one order placed by a customer.

In [17]:
# Drop orders table if it already exists

cursor.execute(

'''

DROP TABLE IF EXISTS orders

'''

)

conn.commit()

In [18]:
create_orders = """

CREATE TABLE IF NOT EXISTS orders(

    order_id TEXT,

    order_date TEXT,

    ship_date TEXT,

    ship_mode TEXT,

    customer_id TEXT,

    product_id TEXT,

    sales REAL,

    quantity INTEGER,

    discount REAL,

    profit REAL

);

"""

conn.execute(create_orders)

print("Orders table created successfully.")

Orders table created successfully.


In [19]:
# Save the changes made to the database

conn.commit()

In [20]:
# Insert Data

conn.execute("""
             
INSERT INTO orders

SELECT DISTINCT

        "Order ID",
        
        "Order Date",
        
        "Ship Date",
        
        "Ship Mode",
        
        "Customer ID",
        
        "Product ID",
        
        Sales,
        
        Quantity,
        
        Discount,
        
        Profit
        
FROM superstore_raw;
                     
""")

In [21]:
# Save the inserted records

conn.commit()

In [22]:
# Display first five records from orders table

pd.read_sql(
    
"""
SELECT *

FROM orders

LIMIT 5;

""",

conn)

,order_id,order_date,ship_date,ship_mode,customer_id,product_id,sales,quantity,discount,profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-BO-10001798,261.9600,2,0.00,41.9136
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
2,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,OFF-LA-10000240,14.6200,2,0.00,6.8714
3,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,OFF-ST-10000760,22.3680,2,0.20,2.5164


### Observation

The orders table has been created successfully and populated with transaction records.

Each record stores important order details such as sales, quantity, discount and profit, which will be used in the upcoming SQL analysis.

## Step 12 : Verify All Tables

In this step, we verify that all required tables have been created successfully and contain data.

This ensures that the database setup process has been completed correctly.

In [23]:
# Display total numbers of customers

pd.read_sql(
    """
    SELECT COUNT(*) AS Total_Customers
    
    FROM customers;
    
    """,
    
    conn
    
)

,Total_Customers
0,44190


In [24]:
# Display total number of products

pd.read_sql(
    """
    SELECT COUNT(*) AS Total_Products
    
    FROM products;
    
    """,
    
    conn)

,Total_Products
0,13258


In [25]:
# Display total number of orders

pd.read_sql(
    
"""
SELECT COUNT(*) AS Total_Orders

FROM orders;

""",

conn
    
)

,Total_Orders
0,9993


In [26]:
# Close the database connection

conn.close()

## Analysis

The database contains separate normalized tables for customers, products and orders.

The verification confirms that all required records have been inserted successfully.

This structure reduces redundancy and makes complex SQL queries easier to write and understand.

### Final Observation

This Superstore dataset has been successfully imported into the SQLite database.

The raw dataset has been normalized into three separate tables:

- customers
- products
- orders

Duplicate records have been removed using the DISTINCT keyword.

The database is now ready for further analysis using Subqueries, CTEs, and Window Functions.

The normalized structure created in this notebook will be used throughout the remaining notebooks to solve business problems using advanced SQL concepts.

## Reflection

While creating this database, I understood the importance of normalization and table design.

Separating customer, product and order information into different tables reduces data redundancy and improves query performance.

This setup provides a strong foundation for implementing Subqueries, Common Table Expressions (CTEs) and Window Functions in the upcoming notebooks.